In [ ]:
## imports
import pandas as pd
import kagglehub
import matplotlib.pyplot as plt
import json

### Assumptions

- Timestamps indicating end of the week (Aligning with convention of hourly observations themselves)
- Weeks start with Monday and end on Sunday (ISO)
- Treatment of null values
- Both dhi and dni are mentiond in the task - I've done for DNI
- This is a static historical datset with no need for incremental load

### Data Acquisition

In [ ]:
# Download latest version of dataaset
path = kagglehub.dataset_download("us-doe/tmy3-solar")

print("Path to dataset files:", path)

In [ ]:
## dataset imports
datapath = f"{path}/tmy3.csv"
metapath = f"{path}/TMY3_StationsMeta.csv"
data_df = pd.read_csv(datapath)
stations = pd.read_csv(metapath)

### Exploratory Data Analysis

In [ ]:
data_df.shape

In [ ]:
data_df.head()

In [ ]:
data_df.info()

In [ ]:
# columns of interest
SELECTED_COLUMNS = [
    'Date (MM/DD/YYYY)',
    'Time (HH:MM)',
    'ETR (W/m^2)',
    'ETRN (W/m^2)',
    'GHI (W/m^2)',
    'GHI source',
    'GHI uncert (%)',
    'DNI (W/m^2)',
    'DNI source',
    'DNI uncert (%)',
    'station_number'
]

In [ ]:
# Make narrow working copy to reduce memory demand.
df = data_df.loc[:, SELECTED_COLUMNS].copy()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# Summary stats
df.describe().T

There are some null values for the columns:
    'GHI (W/m^2)'
    'GHI uncert (%)'
    'DNI (W/m^2)'
    'DNI uncert (%)'

In [ ]:
## Missing data percent
df.isnull().mean() * 100

In [ ]:
# Source flag distributions
for col in ['GHI source', 'DNI source']:
    print(f"\nValue counts for {col}:")
    display(df[col].value_counts(dropna=False).head(20))

In [ ]:
# Simple histograms on a sample to avoid plotting millions of points
sample_n = min(300_000, len(df))
sample = df[['GHI (W/m^2)', 'DNI (W/m^2)']].sample(sample_n, random_state=42) if len(df) > sample_n else df[['GHI (W/m^2)', 'DNI (W/m^2)']]

plt.figure(figsize=(10, 4))
plt.hist(sample['GHI (W/m^2)'].dropna(), bins=100)
plt.title("Sample distribution of GHI")
plt.xlabel("GHI (W/m^2)")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(sample['DNI (W/m^2)'].dropna(), bins=100)
plt.title("Sample distribution of DNI")
plt.xlabel("DNI (W/m^2)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
## Distinct station coverage
print("Distinct station_number in data:", df['station_number'].nunique(dropna=True))

In [ ]:
# Average hourly profile by local clock time before UTC conversion
hourly_profile = (
    df.groupby('Time (HH:MM)')[['GHI (W/m^2)', 'DNI (W/m^2)']]
      .mean()
      .sort_index()
)
plt.figure(figsize=(12, 4))
plt.plot(hourly_profile.index.astype(str), hourly_profile['GHI (W/m^2)'])
plt.xticks(rotation=90)
plt.title("Average GHI by local clock time")
plt.xlabel("Time (HH:MM)")
plt.ylabel("Average GHI")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(hourly_profile.index.astype(str), hourly_profile['DNI (W/m^2)'])
plt.xticks(rotation=90)
plt.title("Average DNI by local clock time")
plt.xlabel("Time (HH:MM)")
plt.ylabel("Average DNI")
plt.tight_layout()
plt.show()


#### check for simple invalid data

In [ ]:
# check for null station number
df.loc[df['station_number'].isna()].shape

In [ ]:
# check for negatives (shouldn't be for this data)
df.loc[df['GHI (W/m^2)'] < 0].shape

In [ ]:
df.loc[df['DNI (W/m^2)'] < 0].shape

#### create a datetime column

In [ ]:
# Identify rows where time is 24:00
is_2400 = df['Time (HH:MM)'] == '24:00'

# Replace 24:00 with 00:00 so pandas can parse
time_fixed = df['Time (HH:MM)'].where(~is_2400, '00:00')

# Build datetime string
datetime_str = df['Date (MM/DD/YYYY)'].astype(str) + " " + time_fixed.astype(str)

# Parse datetime
df['parsed_local'] = pd.to_datetime(datetime_str, format="%m/%d/%Y %H:%M", errors="coerce")

# Add one day where the original time was 24:00
df.loc[is_2400, 'parsed_local'] = df.loc[is_2400, 'parsed_local'] + pd.Timedelta(days=1)

In [ ]:
df.head()

In [ ]:
## check for failed datetime casts
df.loc[df['parsed_local'].isna()].shape

In [ ]:
# check for duplicates by station_number, datetime
df[df.duplicated(['station_number','parsed_local'], keep=False)].shape

#### check for outliers by station-month-hour

In [ ]:
df['month'] = df['parsed_local'].dt.month

In [ ]:
group_cols = ['station_number', 'month', 'Time (HH:MM)']

df['ghi_mean'] = df.groupby(group_cols)['GHI (W/m^2)'].transform('mean')
df['ghi_std']  = df.groupby(group_cols)['GHI (W/m^2)'].transform('std')

df['dni_mean'] = df.groupby(group_cols)['DNI (W/m^2)'].transform('mean')
df['dni_std']  = df.groupby(group_cols)['DNI (W/m^2)'].transform('std')

ghi_outlier = (
    df['ghi_std'].notna() &
    (df['ghi_std'] > 0) &
    ((df['GHI (W/m^2)'] - df['ghi_mean']).abs() > 3 * df['ghi_std'])
)

dni_outlier = (
    df['dni_std'].notna() &
    (df['dni_std'] > 0) &
    ((df['DNI (W/m^2)'] - df['dni_mean']).abs() > 3 * df['dni_std'])
)

outliers = df.loc[
    ghi_outlier | dni_outlier,
    [
        'station_number',
        'parsed_local',
        'month',
        'Time (HH:MM)',
        'GHI (W/m^2)',
        'DNI (W/m^2)',
        'ghi_mean',
        'ghi_std',
        'dni_mean',
        'dni_std'
    ]
]

In [ ]:
print("Total outlier rows:", len(outliers))
print("GHI outliers:", ghi_outlier.sum())
print("DNI outliers:", dni_outlier.sum())
outliers.head()

#### Metadata (stations) profiling

In [ ]:
stations.head()

In [ ]:
stations.info()

In [ ]:
print("Distinct USAF in metadata:", stations['USAF'].nunique(dropna=True))

### Join the two datasets

In [ ]:
# Join metadata
station_lookup = stations.rename(columns={
    'USAF': 'station_number',
    'Site Name': 'site_name',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'TZ': 'tz_hours'
})

work = df.merge(
    station_lookup[['station_number', 'site_name', 'latitude', 'longitude', 'tz_hours']],
    on='station_number',
    how='left',
    validate='many_to_one'
)
print(work.shape)
work.head()

In [ ]:

print("Rows with missing metadata after join:", work['site_name'].isna().sum())

#### Convert local standard time to UTC.

In [ ]:
# TMY3 TZ is hours from Greenwich, negative west.
# local = UTC + tz_hours  => UTC = local - tz_hours
work['utc_dt'] = work['parsed_local'] - pd.to_timedelta(work['tz_hours'], unit='h')

work[['station_number', 'parsed_local', 'tz_hours', 'utc_dt']].head()

### Data Quality Handling

Decisions
- remove na/s (small percentage)
- don't remove outliers:
    - I assume based on long tail distribution this is expected
    - I'm not sure what's appropriate for this dataset 
    - I've already confirmed there's no spurious negatives

In [ ]:
work = work[work[['GHI (W/m^2)', 'DNI (W/m^2)']].notna().all(axis=1)]
work.shape

### JSON Output Prep

In [ ]:
# We want weeks that start Monday and end Sunday.
# In pandas Period('W-SUN'), the period ends on Sunday.
# We use the UTC timestamp to assign each observation to a UTC week.

weekly = (
    work.assign(week_period=work['utc_dt'].dt.to_period('W-SUN'))
        .groupby(['station_number', 'site_name', 'latitude', 'longitude', 'week_period'], observed=True, as_index=False)
        .agg(
            ghi=('GHI (W/m^2)', 'mean'),
            dni=('DNI (W/m^2)', 'mean'),
            ghi_obs=('GHI (W/m^2)', lambda s: s.notna().sum()),
            dni_obs=('DNI (W/m^2)', lambda s: s.notna().sum()),
            total_rows=('utc_dt', 'size')
        )
)

# week ending timestamp in UTC
weekly['week_end_utc'] = weekly['week_period'].dt.end_time.dt.tz_localize('UTC')

# Use integer milliseconds since epoch Because pandas stores timestamps internally as nanoseconds
weekly['timestamp'] = (weekly['week_end_utc'].astype('int64') // 10**6).astype('int64')

weekly.head()

In [ ]:
# Prepare station master list for all stations from metadata
all_stations = (
    station_lookup[['station_number', 'site_name', 'latitude', 'longitude']]
    .drop_duplicates()
    .sort_values('station_number')
    .reset_index(drop=True)
)

all_stations.head()

In [ ]:
# Reduce weekly frame to output fields
weekly_out = weekly[['station_number', 'timestamp', 'ghi', 'dni']].copy()

In [ ]:
# Build nested JSON structure
records = []

weekly_grouped = {
    station_id: grp.sort_values('timestamp')
    for station_id, grp in weekly_out.groupby('station_number', observed=True)
}

for row in all_stations.itertuples(index=False):
    station_id = int(row.station_number) if pd.notna(row.station_number) else None
    site_name = None if pd.isna(row.site_name) else str(row.site_name)
    latitude = None if pd.isna(row.latitude) else float(row.latitude)
    longitude = None if pd.isna(row.longitude) else float(row.longitude)

    grp = weekly_grouped.get(station_id)
    data_items = []

    if grp is not None:
        for r in grp.itertuples(index=False):
            data_items.append({
                "timestamp": int(r.timestamp),
                "ghi": None if pd.isna(r.ghi) else float(r.ghi),
                "dni": None if pd.isna(r.dni) else float(r.dni),
            })

    records.append({
        "id": str(station_id) if station_id is not None else None,
        "site_name": site_name,
        "coordinates": [longitude, latitude],   # matches your example ordering [150, -26]
        "data": data_items
    })

len(records), records[0]

In [ ]:
json_str = json.dumps(records, indent=4)
with open("data/output.json", "w") as f:
    f.write(json_str)

### Tests/Assertions

Should verify
- station count in JSON equals station count in meta_df
- every object has keys: id, site_name, coordinates, data
- coordinates exist as [longitude, latitude]
- JSON station ids are unique
- stations with no source data have empty data
- aggregated station-weeks match the weekly dataframe counts
- timestamps are integers in milliseconds and sorted ascending within each station
- no duplicate timestamps within a station
- ghi and dni are either null or non-negative numbers

In [ ]:
with open("data/output.json", "r") as f:
    json_back = json.load(f)

In [ ]:
# Basic structural checks
assert isinstance(json_back, list), "Top-level JSON must be a list"
assert len(json_back) == len(all_stations), "JSON should contain all stations from metadata"

required_station_keys = {"id", "site_name", "coordinates", "data"}
required_data_keys = {"timestamp", "ghi", "dni"}

seen_ids = set()

for station_obj in json_back:
    assert required_station_keys.issubset(station_obj.keys()), f"Missing station keys in {station_obj}"
    assert isinstance(station_obj["coordinates"], list) and len(station_obj["coordinates"]) == 2, f"Bad coordinates for {station_obj.get('id')}"
    assert station_obj["id"] not in seen_ids, f"Duplicate station id {station_obj['id']}"
    seen_ids.add(station_obj["id"])

    timestamps = []
    for item in station_obj["data"]:
        assert required_data_keys.issubset(item.keys()), f"Missing data keys in station {station_obj['id']}"
        assert isinstance(item["timestamp"], int), f"Timestamp must be int for station {station_obj['id']}"
        if item["ghi"] is not None:
            assert isinstance(item["ghi"], (int, float)), f"ghi must be numeric for station {station_obj['id']}"
            assert item["ghi"] >= 0, f"ghi must be non-negative for station {station_obj['id']}"
        if item["dni"] is not None:
            assert isinstance(item["dni"], (int, float)), f"dni must be numeric for station {station_obj['id']}"
            assert item["dni"] >= 0, f"dni must be non-negative for station {station_obj['id']}"
        timestamps.append(item["timestamp"])

    assert timestamps == sorted(timestamps), f"Timestamps not sorted for station {station_obj['id']}"
    assert len(timestamps) == len(set(timestamps)), f"Duplicate timestamps for station {station_obj['id']}"

print("Structural tests passed.")

In [ ]:
# Number of stations with non-empty data should match stations represented in weekly_out
json_non_empty_station_count = sum(1 for s in json_back if len(s["data"]) > 0)
weekly_station_count = weekly_out['station_number'].nunique()

assert json_non_empty_station_count == weekly_station_count, "Mismatch in populated station count"

# Number of weekly records in JSON should match weekly_out row count
json_weekly_record_count = sum(len(s["data"]) for s in json_back)
assert json_weekly_record_count == len(weekly_out), "Mismatch in total weekly record count"

print("Count reconciliation tests passed.")

### Opportunities for extension

- Profile GHI and DNI uncertainty and remove rows where uncertainty is out of tollerance (additional reading and domain knowledge required)
- If you wanted to productionise this for incremental load of updating data a meta store to house watermarks would be useful.
- Automate tests via a deployment pipeline.
- Production processing code may benefit from chunking and/or parallelisation.